# 📝 그래프 알고리즘 과제 LV1(기초): 커뮤니티, 유사도, 경로

> 이 단원의 새 기술을 **하나씩** 확인합니다. 문제는 네 갈래입니다.
>
> - **1. 커뮤니티 탐지**: Leiden 묶음, 크기 분포, 같은 묶음 판별, `write`
> - **2. 알고리즘 견주기**: Louvain·라벨 전파 비교, 가중치와 모듈러리티
> - **3. 노드 유사도**: 노선 상대가 겹치는 정도, `degreeCutoff`
> - **4. 최단 경로**: 몇 편으로 닿는지, 그 길이 지나는 공항, 전원까지의 거리 분포

## 풀이 방법
1. 맨 위 **준비 셀 네 개 + 투영 셀**을 차례로 실행하세요(연결 → 초기화 → 투영 정리 → 적재 → 투영).
2. 각 문제의 **답안 셀**을 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).

- 데이터: **남아시아 항공 노선망**(107공항 · 5개 국가). 노선 관계는 **방향별 488건**입니다(공항 쌍 255개 가운데 왕복 233쌍, 한 방향만 있는 쌍 22개). 속성은 아래 표에 정리해 두었습니다.
- 공항에 붙은 **`country`**(국가·지역)는 참고용으로만 봅니다. 정답과 맞대는 것은 LV2 입니다.
- 커뮤니티 **번호는 실행마다 바뀝니다**. "어느 공항끼리 한 묶음인가"로 읽으세요.

> 출처: OpenFlights `airports.dat`·`routes.dat`(ODbL). 노선은 **2014년 6월에 갱신이 멈춘 자료**라 시간표가 아니라 노선 유무만 담겨 있습니다.

화이팅!

아래 준비 셀들을 먼저 실행하세요. Neo4j 는 반드시 **실습 전용 DB**에 연결합니다.

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 지우기 전에 이 DB 가 맞는지 먼저 확인합니다.
UNIT_LABELS = ['Airport', 'Compound', 'DayType', 'Disease', 'ExDenseEvent', 'ExDisease', 'ExDomestic', 'ExDrug', 'ExEvent', 'ExForeign', 'ExIdKey', 'ExKing', 'ExKinng', 'ExNameKey', 'ExNodeKeyDemo', 'ExPerson', 'ExReign', 'ExScopeEvent', 'ExThrone', 'ExUniqueDemo', 'ExWorld', 'ExYear', 'FlatDrug', 'FlatKing', 'FlatOrder', 'Gene', 'IdKey', 'Line', 'Member', 'NameKey', 'NodeClass', 'NodeDay', 'NodeDrug', 'NodeKing', 'NodeMonth', 'NodeOrder', 'NodeYear', 'Person', 'PharmacologicClass', 'Station', 'SurveyDay', 'SurveyYear', 'Symptom', 'TempStation', 'TryDay', 'TryLine', 'TryStation']   # 이 단원이 만드는 레이블 전부(앞 일차가 남긴 것까지)

# 이 단원 것이 아닌 노드가 하나라도 있으면 지우지 않고 멈춥니다.
# .env 의 주소가 어긋나도 접속은 조용히 성공하므로, 지우기 전에 확인하는 수밖에 없습니다.
# all 로 보는 이유: any 로 보면 :Person:PatientRecord 처럼 한 레이블만 겹치는 남의 노드가 통과합니다
foreign = run_cypher("""
MATCH (n) WHERE size(labels(n)) = 0
   OR NOT all(label IN labels(n) WHERE label IN $unit_labels)
RETURN DISTINCT labels(n) AS labels LIMIT 5""", unit_labels=UNIT_LABELS)
if foreign:
    raise RuntimeError(
        f"{NEO4J_URI} 에 이 단원 것이 아닌 노드가 있습니다: {foreign}\n"
        "다른 실습이나 개인 데이터가 든 DB 로 보여 초기화를 멈췄습니다.\n"
        ".env 의 NEO4J_URI 가 실습 전용 DB 를 가리키는지 먼저 확인하세요.\n"
        "주소가 맞다면 위 레이블은 앞 실습이 남긴 것입니다. UNIT_LABELS 에 더하고 다시 실행하세요.")

# 여기까지 왔으면 이 DB 에는 이 단원이 만든 노드밖에 없습니다.
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH: 노드에 붙은 관계까지 함께 지운다

print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

In [ ]:
# [제공 코드] 남아 있는 GDS 투영(메모리 그래프)을 모두 정리: 이 셀은 실행만 하세요.
# 앞 실습의 사본이 남아 있으면 같은 이름으로 다시 투영할 때 충돌합니다.
for row in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher("CALL gds.graph.drop($g) YIELD graphName", g=row["graphName"])
print("남은 투영:", run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"))

<img src="images/과제/항공_남아시아_구성.png" width="900">

| 레이블 | 뜻 | 속성 |
|---|---|---|
| `:Airport` | 공항 107곳 | `iata` 공항 코드 · `name`·`city` 이름과 도시 · `country` 국가·지역 5종 · `lat`·`lon` 좌표 |

| 관계 | 잇는 것 | 속성 |
|---|---|---|
| `ROUTE` | 공항 → 공항 (방향별 488건) | `km` 두 공항 사이 거리 · `hours` km/800 + 1 · `airlines` 운항 항공사 수 |

> **`km`·`hours` 는 원본에 없는 값입니다.** `km` 은 두 공항 좌표로 계산했고, `hours` 는 `km/800 + 1` 로 정했습니다.

In [ ]:
# [제공 코드] 남아시아 항공 노선망 적재: 이 셀은 실행만 하세요(실습에 쓸 그래프를 만듭니다).
import pandas as pd

airports = pd.read_csv("data/airports_south_asia_nodes.csv")   # iata, name, city, country, lat, lon
routes = pd.read_csv("data/airports_south_asia_edges.csv")     # from_iata, to_iata, km, hours, airlines

# iata 로 공항을 찾을 때 전체를 훑지 않도록 유일성 제약을 먼저 겁니다(인덱스가 함께 생깁니다).
run_cypher("CREATE CONSTRAINT airport_iata IF NOT EXISTS FOR (a:Airport) REQUIRE a.iata IS UNIQUE")

# 공항을 노드로 만듭니다.
run_cypher("""UNWIND $rows AS r
    CREATE (:Airport {iata: r.iata, name: r.name, city: r.city, country: r.country,
                      lat: r.lat, lon: r.lon})""", rows=airports.to_dict("records"))

# 노선을 관계로 잇습니다. 왕복이면 CSV 에 두 행이 있어 관계도 둘이 됩니다.
run_cypher("""UNWIND $rows AS r
    MATCH (a:Airport {iata: r.from_iata}), (b:Airport {iata: r.to_iata})
    CREATE (a)-[:ROUTE {km: r.km, hours: r.hours, airlines: r.airlines}]->(b)""",
           rows=routes.to_dict("records"))

print("공항:", run_cypher("MATCH (a:Airport) RETURN count(a) AS n")[0]["n"],
      "· 노선 관계(방향별):", run_cypher("MATCH (:Airport)-[r:ROUTE]->() RETURN count(r) AS n")[0]["n"])

In [ ]:
# [제공 코드] 남아시아 항공 노선망을 무방향으로 투영합니다: 이 셀은 실행만 하세요.
# 기존 air 투영이 있다면 제거합니다.
run_cypher("CALL gds.graph.drop('air', false) YIELD graphName")

# 노선은 방향이 있지만 '어느 공항끼리 이어져 있는가'에는 방향이 의미가 없어 UNDIRECTED 로 만듭니다.
stats = run_cypher('''
    CALL gds.graph.project('air', 'Airport',
      {ROUTE: {orientation: 'UNDIRECTED', properties: ['km', 'hours', 'airlines']}})
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount''')
print(stats[0])

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 노선망의 규모와 상대 공항이 많은 곳을 먼저 확인합니다.

In [ ]:
# [제공 코드] 노선망 규모, 항공사 수 분포, 상대 공항이 많은 공항
import pandas as pd

print('공항 수:', run_cypher('MATCH (a:Airport) RETURN count(a) AS n')[0]['n'])
print('노선 관계 수(방향별):',
      run_cypher('MATCH (:Airport)-[r:ROUTE]->() RETURN count(r) AS n')[0]['n'])
# airlines 는 그 방향을 운항하는 항공사 수입니다. 2-3 에서 가중치로 넘깁니다
display(pd.DataFrame(run_cypher('''
    MATCH ()-[r:ROUTE]->()
    RETURN r.airlines AS airlines, count(*) AS routes
    ORDER BY airlines''')))
display(pd.DataFrame(run_cypher('''
    MATCH (a:Airport)
    OPTIONAL MATCH (a)-[:ROUTE]-(other:Airport)
    RETURN a.iata AS iata, a.city AS city, a.country AS country,
           count(DISTINCT other) AS partners
    ORDER BY partners DESC LIMIT 5''')))

---
# 1. 커뮤니티 탐지

Leiden 으로 묶음을 찾고, 크기를 세고, 노드에 저장합니다(교안_01 2절).

## 1-1. Leiden 으로 커뮤니티 찾기
**배경**: 자주 오가는 공항끼리 생기는 촘촘한 묶음을 Leiden 으로 찾습니다.

**요구사항**:
- `gds.leiden.stream` 을 `'air'` 투영에 **가중치 없이** 호출하세요.
- 결과를 리스트 **`comm_rows`** 에 담으세요(각 행에 `iata`(공항 코드)·`c`(커뮤니티 번호) 두 열이 있게 별칭을 붙이세요).
- 커뮤니티 번호의 종류 수를 **`n_comm`** 에 담으세요.

**예시**: `comm_rows` 는 107줄, `n_comm` 은 **한 자리 수**입니다(번호와 개수는 실행마다 달라질 수 있습니다).

<details><summary>힌트</summary>

```text
접근방법:
- Leiden 을 stream 으로 돌려 커뮤니티 번호를 받고, 파이썬에서 종류를 센다.

세부구현:
1. 커뮤니티 탐지를 stream 으로 호출해 노드 id 와 커뮤니티 id 를 YIELD 로 받는다.
2. 노드 id 는 GDS 내부 번호다. gds.util.asNode 로 되돌려 공항 코드에 iata,
   커뮤니티 id 에 c 별칭을 붙인다.
3. 결과를 comm_rows 에, c 값의 집합(set) 크기를 n_comm 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 개수는 실행마다 흔들리므로 범위로, 조회를 실제로 돌렸는지는 줄 수와 공항 코드로 봅니다
assert len(comm_rows) == 107, \
    'comm_rows 는 한 공항당 한 줄, 총 107줄이어야 합니다. stream 결과를 그대로 담았는지 확인하세요'
assert 'iata' in comm_rows[0] and 'c' in comm_rows[0], \
    '각 행에 iata 와 c 별칭이 있어야 합니다. RETURN 절의 별칭을 확인하세요'
assert len({r['c'] for r in comm_rows}) == n_comm, \
    'n_comm 은 comm_rows 의 c 값 종류 수여야 합니다. 손으로 적지 말고 집합 크기로 구하세요'
assert 3 <= n_comm <= 10, \
    f'커뮤니티가 {n_comm}개로 나왔습니다. air 투영에 Leiden 을 돌렸는지 확인하세요'
# 공항 코드가 DB 의 107곳과 정확히 같은지 본다. 손으로 지어낸 값이면 여기서 걸린다
_codes = {r['iata'] for r in run_cypher('MATCH (a:Airport) RETURN a.iata AS iata')}
assert {r['iata'] for r in comm_rows} == _codes, \
    'comm_rows 의 공항 코드가 DB 의 공항과 다릅니다. 조회 결과를 그대로 담았는지 확인하세요'
print('✅ 통과!')

## 1-2. 커뮤니티 크기 분포 구하기
**배경**: 한 덩어리가 전체를 삼켰는지 크기로 확인합니다.

**요구사항**:
- 1-1 의 `comm_rows` 로 커뮤니티별 공항 수를 세어 **큰 것부터 정렬한 리스트** **`comm_sizes`** 를 만드세요(원소는 정수 공항 수). 길이가 `n_comm` 과 같아야 하니 **1-1 을 먼저 푸세요.**
- 가장 큰 커뮤니티의 공항 수를 **`max_size`** 에 담으세요.

**예시**: `comm_sizes` 는 `[가장 큰 묶음, 그다음, ...]` 모양의 **정수 리스트**이고 합계는 107 입니다(값도 개수도 실행마다 달라집니다).

<details><summary>힌트</summary>

```text
접근방법:
- 커뮤니티 번호별로 몇 곳인지 세고, 그 값만 뽑아 내림차순으로 정렬한다.

세부구현:
1. comm_rows 로 커뮤니티 번호마다 공항 수를 센다(collections 의 개수 세는 도구든 사전이든 좋다).
2. 값(공항 수)만 꺼내 내림차순으로 정렬해 comm_sizes 에, 첫 원소를 max_size 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 합계는 정확히, 최대 크기는 흔들림을 감안해 범위로 확인합니다
assert sum(comm_sizes) == 107, \
    f'공항 합계가 {sum(comm_sizes)} 입니다. 107곳 전부가 어느 묶음엔가 들어가야 합니다'
assert list(comm_sizes) == sorted(comm_sizes, reverse=True), \
    'comm_sizes 는 큰 것부터 정렬돼야 합니다'
assert len(comm_sizes) == n_comm, \
    'comm_sizes 의 길이는 커뮤니티 개수와 같아야 합니다'
assert max_size == comm_sizes[0] and 15 <= max_size <= 70, \
    f'가장 큰 커뮤니티가 {max_size}곳입니다. 한 덩어리로 다 묶였다면 알고리즘 호출을 확인하세요'
print('✅ 통과!')

## 1-3. 두 공항이 같은 묶음인지 판별하기
**배경**: 번호는 실행마다 바뀌지만 "어느 공항끼리 한 묶음인가"는 안정적입니다.

**요구사항**:
- 1-1 의 결과로 **`{공항 코드: 커뮤니티 번호}`** 사전 **`comm_of`** 를 만드세요.
- 두 공항이 같은 커뮤니티인지 돌려주는 함수 **`same_group(a, b)`** 를 만드세요(bool 반환).
- `same_group('BOM', 'DEL')` 와 `same_group('BOM', 'KHI')` 를 각각 출력하세요.

**예시**: 뭄바이(BOM) 와 델리(DEL) 는 `True`, 뭄바이(BOM) 와 카라치(KHI) 는 `False` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 공항 코드가 열쇠, 커뮤니티 번호가 값인 사전을 만든 뒤 두 값을 비교한다.

세부구현:
1. 1-1 에서 받은 행들로 사전을 만든다(사전 컴프리헨션을 써도 좋다).
2. same_group 은 인자 두 개를 받아 사전에서 각각의 번호를 꺼낸다.
3. 두 번호가 같은지 비교한 결과(bool)를 돌려준다(크기가 아니라 '같은가'만 본다).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 번호가 아니라 '같은 묶음인가'로 비교했는지 확인합니다
assert len(comm_of) == 107, \
    f'comm_of 에 {len(comm_of)}곳만 들어 있습니다. 107곳 전부가 있어야 합니다'
assert bool(same_group('BOM', 'DEL')) is True, \
    'BOM 와 DEL 는 늘 같은 묶음입니다. 사전에서 번호를 꺼내 비교했는지 확인하세요'
assert bool(same_group('BOM', 'KHI')) is False, \
    'BOM 와 KHI 는 다른 묶음입니다. 두 값을 == 로 비교했는지 확인하세요'
# 공항 코드가 DB 와 정확히 같은지 본다. 손으로 지어낸 값이면 여기서 걸린다
_codes = {r['iata'] for r in run_cypher('MATCH (a:Airport) RETURN a.iata AS iata')}
assert set(comm_of) == _codes, \
    'comm_of 의 열쇠가 DB 의 공항 코드와 다릅니다. comm_rows 에서 만들었는지 확인하세요'
print('✅ 통과!')

## 1-4. 커뮤니티를 노드 속성으로 저장하기
**배경**: 저장해 두면 GDS 없이 평범한 Cypher 로도 커뮤니티를 씁니다.

**요구사항**:
- `gds.leiden.write` 로 `'air'` 투영의 커뮤니티를 **`group_id`** 라는 속성에 저장하세요.
- 저장된 노드 수를 **`n_written`** 에 담으세요(`nodePropertiesWritten`).
- **다시 조회해** `group_id` 가 있는 공항 수를 **`n_has_group`** 에 담으세요.

**예시**: `n_written` 과 `n_has_group` 은 모두 **107** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- stream 대신 write 로 호출하고, 저장 뒤에 일반 Cypher 로 확인한다.

세부구현:
1. 커뮤니티 탐지를 write 로 호출한다.
   1-1. 설정 맵에 저장할 속성 이름을 넣는다(키 이름은 writeProperty).
   1-2. YIELD 로 저장된 노드 수를 받아 n_written 에 담는다.
2. 일반 Cypher 로 그 속성이 있는 노드를 세어 n_has_group 에 담는다(조건은 IS NOT NULL).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 값뿐 아니라 DB 에 실제로 쓰였는지 다시 조회해 확인합니다
assert n_written == 107, \
    f'저장된 노드가 {n_written}개입니다. 107곳 전부에 써야 합니다'
assert n_has_group == 107, \
    'group_id 속성이 그래프에 남지 않았습니다. writeProperty 이름을 확인하세요'
_probe = run_cypher('MATCH (a:Airport) RETURN count(DISTINCT a.group_id) AS groups')[0]['groups']
assert 3 <= _probe <= 10, \
    f'group_id 종류가 {_probe}가지입니다. 커뮤니티 번호가 아니라 다른 값을 저장했는지 확인하세요'
print('✅ 통과!')

---
# 2. 알고리즘 견주기

같은 그래프에 다른 알고리즘을 돌려 무엇이 달라지는지 봅니다(교안_01 4절·5-2).

## 2-1. Louvain 으로 돌려 개수 비교하기
**배경**: 같은 그래프에 다른 알고리즘을 돌려 봅니다.

**요구사항**:
- `gds.louvain.stream` 을 `'air'` 투영에 호출하세요.
- 커뮤니티 번호별 공항 수를 센 결과를 **`louvain_counted`** 에 담으세요(`{커뮤니티 번호: 공항 수}` 모양이면 `Counter` 든 보통 사전이든 상관없습니다).
- 그 종류 수를 **`n_louvain`**, 가장 큰 값을 **`max_louvain`** 에 담으세요.

**예시**: `n_louvain` 은 **한 자리 수**, `max_louvain` 은 전체의 **절반이 안 됩니다**(실행마다 흔들리고, 1-1 의 Leiden 결과와 크게 다르지 않습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 1-1 과 같되 알고리즘 이름만 바꾸고, 센 결과·개수·최대 공항 수를 구한다.

세부구현:
1. Louvain 을 stream 으로 호출해 노드마다 커뮤니티 번호를 받는다.
2. 번호별 공항 수를 세어 louvain_counted 에 담는다(1-1 과 같은 방법).
3. 그 종류 수를 n_louvain, 값 중 최댓값을 max_louvain 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 개수, 최대 크기는 범위로, 합계는 정확히 확인합니다
assert sum(louvain_counted.values()) == 107, \
    '공항 합계가 107곳이어야 합니다. 결과를 거르지 않았는지 확인하세요'
assert 3 <= n_louvain <= 10, \
    f'Louvain 커뮤니티가 {n_louvain}개로 나왔습니다. air 투영에 돌렸는지 확인하세요'
assert 15 <= max_louvain <= 68, \
    f'가장 큰 커뮤니티가 {max_louvain}곳입니다. 107 이라면 전부를 한 덩어리로 묶은 것입니다'
# Louvain 은 실행마다 답이 달라 라이브 대조를 못 한다. 대신 두 값이 표에서 나온 것인지 본다
assert n_louvain == len(louvain_counted) and max_louvain == max(louvain_counted.values()), \
    '두 값은 louvain_counted 에서 세어야 합니다. 손으로 적은 값이 아닌지 확인하세요'
print('✅ 통과!')

## 2-2. 라벨 전파는 얼마나 큰 덩어리를 만드는지 재기
**배경**: 라벨 전파는 이웃의 다수결로 퍼뜨려 **한 라벨이 크게 번지기 쉽고**, 동점일 때의 규칙이 없어 **적재할 때마다 답이 달라집니다.** 개수만 보지 말고 **가장 큰 묶음이 전체의 몇 할인지**를 함께 재야 합니다.

**요구사항**:
- `gds.labelPropagation.stream` 을 `'air'` 투영에 호출하세요.
- 2-1 과 같은 모양으로 커뮤니티 번호별 공항 수를 **`lp_counted`** 에 담으세요.
- 그 종류 수를 **`n_lp`**, 가장 큰 값을 **`biggest_lp`** 에 담으세요.
- 가장 큰 묶음이 전체 107곳에서 차지하는 비율을 소수 **둘째 자리까지 반올림**해 **`biggest_ratio`** 에 담으세요.
- 1-1·1-2 의 Leiden 결과와 나란히 출력해 견주어 보세요.

**예시**: `biggest_ratio` 는 **0 과 1 사이의 소수**입니다. 묶음 수는 Leiden 보다 **적게 나오기 쉽고** 그만큼 한 묶음이 큽니다. **다시 적재해 돌리면 다른 답이 나오니** 한 번 돌린 값으로 단정하지 마세요.

<details><summary>힌트</summary>

```text
접근방법:
- 2-1 과 같되 알고리즘 이름만 바꾸고, 비율 한 가지를 더 구한다.

세부구현:
1. 라벨 전파를 stream 으로 호출해 번호별 공항 수를 세어 lp_counted 에 담는다.
2. 그 종류 수를 n_lp, 값 중 최댓값을 biggest_lp 에 담는다.
3. 최댓값을 전체 공항 수로 나누고 소수 둘째 자리로 반올림해 biggest_ratio 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 합계는 정확히, 묶음 수와 비율은 범위로 확인합니다
assert sum(lp_counted.values()) == 107, \
    '공항 합계가 107곳이어야 합니다'
# 라벨 전파는 적재(내부 번호 순서)마다 답이 갈립니다. 지금까지 관측한 폭은 묶음 1~4개, 최대 묶음 비율 0.71~1.0 인데,
# 다른 적재에서 더 벌어질 수 있어 채점은 그 폭보다 훨씬 넓게 잡습니다(합계만 정확히 봅니다).
assert 1 <= n_lp <= 6, \
    f'묶음이 {n_lp}개로 나왔습니다. air 투영에 gds.labelPropagation 을 돌렸는지 확인하세요'
assert biggest_ratio == round(biggest_lp / 107, 2), \
    'biggest_ratio 는 가장 큰 묶음을 전체 공항 수로 나눈 값이어야 합니다'
assert 0.3 <= biggest_ratio <= 1.0, \
    f'가장 큰 묶음이 전체의 {biggest_ratio} 입니다. 커뮤니티별 공항 수의 최댓값을 썼는지 확인하세요'
# 여기서 다시 돌려 묶음 수가 비슷하게 나오는지 본다. 손으로 적은 값이면 여기서 걸린다
_live_lp = run_cypher("CALL gds.labelPropagation.stats('air') "
                      "YIELD communityCount RETURN communityCount")[0]['communityCount']
assert abs(n_lp - _live_lp) <= 1, \
    f'다시 재 보니 {_live_lp}개인데 n_lp 는 {n_lp} 입니다. 직접 돌린 결과를 담으세요'
print('✅ 통과!')

## 2-3. 항공사 수를 반영하면 묶음이 뚜렷해지는지 재기
**배경**: 노선을 **몇 개 항공사가 운항하는지**(`airlines`)까지 반영하면 어떻게 달라지는지 **모듈러리티**로 잽니다.

**요구사항**:
- `gds.leiden.stats` 를 `'air'` 투영에 두 번 호출해 모듈러리티를 구하세요.
  - 가중치 없이 구한 값을 **`mod_plain`** 에 담으세요.
  - `relationshipWeightProperty` 에 `'airlines'` 를 준 값을 **`mod_weighted`** 에 담으세요.
- 둘 다 흔들림을 없애려고 `randomSeed: 42, concurrency: 1` 을 함께 주세요(교안_01 5-2).
- 소수 넷째 자리까지 반올림해 담고, 두 값을 함께 출력해 **어느 쪽이 큰지** 확인하세요.

**예시**: 두 값 모두 **0 과 1 사이의 소수**이고, 어느 쪽이 큰지는 **재 봐야 압니다.** 가중치를 실으면 값이 오를 것 같지만 실제 방향을 확인하고 **왜 그런지 한두 문장으로 생각해 보세요**(정답 노트북에 해설이 있습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 같은 프로시저를 stats 로 두 번 부르되, 두 번째에만 가중치 속성 이름을 설정에 넣는다.

세부구현:
1. 커뮤니티 탐지를 stats 로 호출한다.
   1-1. 설정 맵에 randomSeed 와 concurrency 를 넣어 결과를 고정한다.
   1-2. YIELD 로 모듈러리티를 받는다.
2. 가중치로 쓸 관계 속성 이름을 하나 더 넣어 다시 부른다.
3. 두 값을 각각 소수 넷째 자리로 반올림해 담는다(round 는 Cypher·파이썬 어느 쪽이든 좋다).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 두 값의 범위와 대소 관계를 확인하고, 같은 자리에서 다시 재 대조합니다
assert 0.4 < mod_plain < 0.53, \
    f'mod_plain 이 {mod_plain} 입니다. 가중치를 주지 않은 쪽인지 확인하세요'
assert 0.33 < mod_weighted < 0.45, \
    f'mod_weighted 가 {mod_weighted} 입니다. relationshipWeightProperty 에 airlines 를 넘겼는지 확인하세요'
assert round(mod_plain, 4) == mod_plain and round(mod_weighted, 4) == mod_weighted, \
    '소수 넷째 자리까지 반올림해 담으세요'
assert mod_weighted < mod_plain, \
    f'가중치를 준 값이 {mod_weighted}, 주지 않은 값이 {mod_plain} 입니다. 두 값을 바꿔 담지 않았는지, airlines 를 정말 가중치로 넘겼는지 확인하세요'
# randomSeed 를 고정했으므로 여기서 다시 재도 같은 값이 나온다. 두 값을 각각 대조한다
_live_plain = run_cypher('''
    CALL gds.leiden.stats('air', { randomSeed: 42, concurrency: 1 })
    YIELD modularity RETURN round(modularity, 4) AS m''')[0]['m']
assert mod_plain == _live_plain, \
    f'mod_plain 이 {mod_plain} 인데 다시 잰 값과 다릅니다. 가중치 없이 부른 결과를 그대로 담았는지, randomSeed 와 concurrency 를 같이 줬는지 확인하세요'
# 가중치 쪽도 같은 자리에서 다시 잰다. 이 대조가 있어야 relationshipWeightProperty 를 정말 넘긴 답안만 통과한다
_live_weighted = run_cypher('''
    CALL gds.leiden.stats('air', { randomSeed: 42, concurrency: 1,
                                   relationshipWeightProperty: 'airlines' })
    YIELD modularity RETURN round(modularity, 4) AS m''')[0]['m']
assert mod_weighted == _live_weighted, \
    f'mod_weighted 가 {mod_weighted} 인데 다시 잰 값과 다릅니다. relationshipWeightProperty 로 airlines 를 넘겼는지, randomSeed 와 concurrency 를 같이 줬는지 확인하세요'
print('✅ 통과!')

---
# 3. 노드 유사도

노선 상대가 겹치는 정도로 닮은 공항을 찾습니다(교안_02 1절).

## 3-1. 두 공항의 노선 상대가 얼마나 겹치는지 재기
**배경**: 노드 유사도(Jaccard)는 "두 공항 모두와 이어진 공항 수"를 "둘 중 하나와라도 이어진 공항 수"로 나눈 값입니다.

**요구사항**:
- `gds.nodeSimilarity.stream` 을 `'air'` 투영에 `topK: 5` 로 호출하세요.
- 결과에서 **델리(DEL) 와 뭄바이(BOM)** 가 짝을 이루는 행만 남겨 **리스트** **`sim_rows`** 에 담으세요(각 행에 공항 코드 `airport`·`partner` 와 유사도 `similarity`(소수 넷째 자리 반올림) 세 열).
- `sim_rows` 첫 행의 유사도를 **`sim_value`** 에 담으세요.

**예시**: `sim_value` 는 **0 과 1 사이의 소수**이고, 이 두 공항은 노선 상대가 **절반 넘게** 겹칩니다(Jaccard 는 결정적이라 다시 실행해도 값이 같습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 유사도 프로시저를 stream 으로 부른 뒤, 그 두 공항이 짝을 이루는 행만 남긴다.

세부구현:
1. 노드 유사도를 stream 으로 호출해 YIELD 로 노드 두 개와 유사도를 받는다.
2. WITH 로 두 노드 id 를 각각 노드로 되돌린다.
3. WHERE 로 양쪽 공항 코드가 원하는 두 값인 행만 남긴다.
4. 두 공항 코드와 반올림한 유사도를 세 열로 받아 sim_rows 에, 첫 행의 유사도를 sim_value 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] Jaccard 는 결정적이라 정확한 값으로 채점합니다
assert sim_value == 0.5932, \
    f'유사도가 {sim_value} 로 나왔습니다. DEL 와 BOM 짝을 골랐는지, 소수 넷째 자리로 반올림했는지 확인하세요'
assert 1 <= len(sim_rows) <= 2, \
    f'sim_rows 가 {len(sim_rows)}줄입니다. 두 공항이 짝을 이루는 행만 남아야 합니다(양쪽 방향을 다 받으면 두 줄). WHERE 조건을 확인하세요'
# 교집합/합집합을 순수 Cypher 로 직접 세어 GDS 가 준 값과 맞는지 본다
_hand = run_cypher('''
    MATCH (a:Airport { iata: $a })--(x:Airport)
    WITH collect(DISTINCT x) AS A
    MATCH (b:Airport { iata: $b })--(y:Airport)
    WITH A, collect(DISTINCT y) AS B
    WITH size([n IN A WHERE n IN B]) AS common, size(A) AS na, size(B) AS nb
    RETURN round(toFloat(common) / (na + nb - common), 4) AS j''',
    a='DEL', b='BOM')[0]['j']
assert sim_value == _hand, \
    f'sim_value 가 {sim_value} 입니다. 두 공항의 공통 노선 상대 수와 합집합 크기를 직접 세어 비교해 보세요'
# 값만 맞고 행이 비어 있으면 조회를 안 한 것이다. 각 행이 그 두 공항의 행인지 본다
assert all({r.get('airport'), r.get('partner')} == {'DEL', 'BOM'} for r in sim_rows), \
    'sim_rows 의 각 행에 DEL 와 BOM 가 airport·partner 로 들어 있어야 합니다'
assert all(r.get('similarity') == 0.5932 for r in sim_rows), \
    'sim_rows 의 similarity 열도 반올림한 값이어야 합니다'
print('✅ 통과!')

## 3-2. 노선이 적은 공항을 빼고 비교하기
**배경**: 노선 상대가 한두 곳뿐인 공항은 그 상대를 공유하는 순간 유사도가 1.0 이 되므로 `degreeCutoff` 로 걸러 냅니다.

**요구사항**:
- `gds.nodeSimilarity.stats` 를 `topK: 1` 로 두 번 호출해 `nodesCompared` 를 받으세요.
  - `degreeCutoff` 를 **1** 로 준 값을 **`compared_all`** 에 담으세요.
  - `degreeCutoff` 를 **10** 으로 준 값을 **`compared_10`** 에 담으세요.

**예시**: `compared_all` 은 **107**(전원), `compared_10` 은 그보다 **훨씬 적습니다**. 상대가 10곳 미만인 작은 공항이 아주 많기 때문입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 프로시저를 stats 로 두 번 부르되 걸러 낼 최소 이웃 수만 다르게 준다.

세부구현:
1. 노드 유사도를 stats 로 호출한다.
   1-1. 설정 맵에 상위 몇 곳까지 볼지와, 이웃이 몇 곳 미만이면 뺄지를 넣는다.
   1-2. YIELD 로 비교 대상이 된 노드 수를 받는다.
2. 최소 이웃 수만 바꿔 한 번 더 부르고, 두 값을 각각 변수에 담아 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 두 값 모두 결정적이라 정확한 값으로 채점합니다
assert compared_all == 107, \
    f'compared_all 이 {compared_all} 입니다. degreeCutoff 를 1 로 줬는지 확인하세요'
assert compared_10 == 11, \
    f'compared_10 이 {compared_10} 입니다. degreeCutoff 를 10 으로 줬는지 확인하세요'
assert compared_10 < compared_all, \
    'degreeCutoff 를 올리면 비교 대상이 줄어야 합니다. 두 값을 바꿔 담지 않았는지 확인하세요'
# stats 로 다시 세어 본다. 유사도 계산은 결정적이라 값이 흔들리지 않는다
_live_10 = run_cypher("CALL gds.nodeSimilarity.stats('air', { degreeCutoff: 10 }) "
                      "YIELD nodesCompared RETURN nodesCompared")[0]['nodesCompared']
assert compared_10 == _live_10, \
    f'compared_10 이 {compared_10} 입니다. degreeCutoff 를 10 으로 준 호출 결과를 그대로 담았는지 확인하세요'
print('✅ 통과!')

---
# 4. 최단 경로

두 공항 사이가 몇 편인지 재고, 그 길이 지나는 공항을 봅니다(교안_02 2절).

## 4-1. 두 공항 사이가 몇 편인지 재기
**배경**: 가중치를 주지 않으면 관계 하나가 비용 1 이 되어, 총비용이 곧 "비행기를 몇 번 타야 하나"인 **편 수**가 됩니다.

**요구사항**:
- `gds.shortestPath.dijkstra.stream` 으로 **델리(DEL) 에서 뭄바이(BOM)** 까지의 최단 경로를 구하세요(가중치는 주지 마세요).
- `totalCost` 를 정수로 바꿔 **`hops`** 에 담으세요.
- 같은 방식으로 **델리(DEL) 에서 조브(PZH)** 까지도 구해 **`far_hops`** 에 담으세요.

**예시**: `hops` 와 `far_hops` 모두 **한 자리 수**이고, `far_hops` 가 더 큽니다.

<details><summary>힌트</summary>

```text
접근방법:
- 출발 노드와 도착 노드를 먼저 MATCH 로 찾아 프로시저에 노드 자체를 넘긴다.

세부구현:
1. MATCH 로 두 공항 노드를 공항 코드로 찾는다.
2. 최단 경로를 stream 으로 호출한다.
   2-1. 설정 맵에 출발 노드와 도착 노드를 넣는다(번호가 아니라 노드 자체다).
   2-2. 가중치 속성은 넣지 않는다. 그래야 총비용이 곧 편 수다.
3. YIELD 로 총비용을 받아 정수로 바꿔 담고, 공항 코드만 바꿔 한 번 더 구한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 편 수는 결정적이라 정확한 값으로 채점합니다
assert hops == 1, \
    f'hops 가 {hops} 로 나왔습니다. 가중치를 주면 편 수가 아니라 거리가 됩니다'
assert far_hops == 3, \
    f'far_hops 가 {far_hops} 로 나왔습니다. DEL 에서 PZH 까지를 쟀는지 확인하세요'
assert isinstance(hops, int) and isinstance(far_hops, int), \
    'totalCost 는 실수로 나옵니다. int() 로 바꿔 담으세요'
# 두 쌍을 여기서 다시 재 본다. 최단 거리는 결정적이라 값이 흔들리지 않는다
def _live_hops(a, b):
    return int(run_cypher('''
        MATCH (x:Airport { iata: $a }), (y:Airport { iata: $b })
        CALL gds.shortestPath.dijkstra.stream('air', { sourceNode: x, targetNode: y })
        YIELD totalCost RETURN totalCost AS t''', a=a, b=b)[0]['t'])


assert hops == _live_hops('DEL', 'BOM') \
    and far_hops == _live_hops('DEL', 'PZH'), \
    f'hops 가 {hops}, far_hops 가 {far_hops} 입니다. 다시 재 본 값과 다릅니다. 직접 조회한 결과를 담았는지 확인하세요'
print('✅ 통과!')

## 4-2. 경로가 지나는 공항 확인하기
**배경**: 편 수만으로는 부족합니다. **어디를 거쳐 가는지**가 실제로 쓰이는 정보입니다.

**요구사항**:
- 4-1 의 **델리(DEL) 에서 조브(PZH)** 까지 경로에서 `nodeIds` 를 받아, 지나는 공항의 **코드 리스트**를 **`route`** 에 담으세요(출발과 도착을 포함합니다).

**예시**: `route` 는 `['DEL', ..., 'PZH']` 모양이고, 길이는 `far_hops` 보다 **하나 많습니다**(출발과 도착을 모두 넣으므로). **가운데 공항은 실행마다 달라질 수 있습니다.**

<details><summary>힌트</summary>

```text
접근방법:
- 4-1 과 같은 호출에서 노드 id 목록도 함께 받아, 리스트 안에서 공항 코드로 바꾼다.

세부구현:
1. 4-1 과 같은 방식으로 최단 경로를 구하되 노드 id 목록도 YIELD 로 받는다.
2. RETURN 절에서 리스트 컴프리헨션으로 각 노드 id 를 되돌려 공항 코드를 꺼낸다.
   - Cypher 의 리스트 컴프리헨션은 [x IN 목록 | 바꿀식] 모양이다.
3. 첫 행의 값을 route 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 양끝, 길이와 함께, 이웃끼리 실제로 노선이 있는지도 확인합니다
assert route[0] == 'DEL' and route[-1] == 'PZH', \
    f'경로가 {route[0]} 에서 시작해 {route[-1]} 에서 끝납니다. DEL 에서 PZH 까지여야 합니다'
assert len(route) == 4, \
    f'경로에 공항이 {len(route)}곳 들어 있습니다. 출발과 도착을 모두 넣으므로 편 수보다 하나 많아야 합니다'
for _x, _y in zip(route, route[1:]):
    _linked = run_cypher('''
        MATCH (a:Airport { iata: $x })
        RETURN EXISTS { (a)-[:ROUTE]-(:Airport { iata: $y }) } AS linked''',
        x=_x, y=_y)[0]['linked']
    assert _linked, f'{_x} 와 {_y} 사이에는 노선이 없습니다. 조회 결과를 그대로 담았는지 확인하세요'
print('✅ 통과!')

## 4-3. 한 공항에서 전원까지의 거리 분포 재기
**배경**: **한 공항에서 전체까지**를 한 번에 재면 이 노선망이 얼마나 좁은지 한눈에 들어옵니다.

**요구사항**:
- `gds.allShortestPaths.dijkstra.stream` 으로 **뭄바이(BOM) 에서 닿는 공항 전부**까지의 거리를 구하세요(도착지를 정하지 않습니다. 가중치도 주지 마세요).
- **편 수를 키, 그 거리에 있는 공항 수를 값**으로 하는 딕셔너리를 **`spread`** 에 담으세요. **키와 값 모두 정수**입니다.
- 출발점 자기 자신도 거리 0 으로 들어 있습니다. **빼지 말고 그대로 두세요.**
- 가장 먼 공항까지의 거리를 **`farthest`** 에 정수로 담으세요.

**예시**: `spread` 는 `{0: 1, 1: ..., 2: ...}` 처럼 거리 0 부터 이어지는 칸이 나오고 값의 합은 107 입니다. `farthest` 는 **한 자리 수**입니다.

> 뭄바이(BOM) 는 **상대 공항이 가장 많은 곳**입니다(델리(DEL) 보다도 많습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 도착 노드를 빼고 출발 노드만 주면 닿는 공항 전부까지의 거리가 한 번에 나온다.
  거리별로 세는 것은 Cypher 집계로 해도 되고 파이썬으로 해도 된다.

세부구현:
1. MATCH 로 출발 공항 노드를 코드로 찾는다.
2. 전부까지의 거리를 구하는 프로시저를 stream 으로 호출한다.
   2-1. 설정 맵에 출발 노드만 넣는다(도착 노드는 넣지 않는다).
   2-2. 가중치 속성은 넣지 않는다. 그래야 총비용이 곧 편 수다.
3. 총비용을 정수로 바꿔 거리별로 세어 딕셔너리로 만든다.
4. 키 중 가장 큰 값을 farthest 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점] 분포는 결정적입니다. 합계와, 1편 거리의 공항 수가 실제 상대 수와 같은지도 봅니다
assert spread == {0: 1, 1: 48, 2: 54, 3: 4}, \
    f'spread 가 {spread} 입니다. 거리 0(자기 자신)까지 넣었는지, 가중치를 주지 않았는지 확인하세요'
assert sum(spread.values()) == 107, \
    f'합계가 {sum(spread.values())}곳입니다. 107곳 전부가 들어가야 합니다'
assert farthest == 3 and isinstance(farthest, int), \
    f'farthest 가 {farthest} 입니다. 키 중 가장 큰 값을 정수로 담으세요'
_partners = run_cypher('''
    MATCH (a:Airport { iata: $a })--(o:Airport)
    RETURN count(DISTINCT o) AS n''',
    a='BOM')[0]['n']
assert spread[1] == _partners, \
    f'1편 거리가 {spread[1]}곳인데 실제 상대 공항은 {_partners}곳입니다. 관계 수가 아니라 공항 수로 세었는지 확인하세요'
print('✅ 통과!')

---
수고했어요! 커뮤니티(Leiden·Louvain·라벨 전파)·묶음 판별·`write`·모듈러리티·노드 유사도·`degreeCutoff`·최단 경로·거리 분포를 **하나씩** 익혔습니다. LV2 에서는 이것들을 **조합**하고, 국가·지역 정답과 대조해 결과가 얼마나 믿을 만한지까지 다룹니다.